# Task 4: Statistical Modeling & Risk-Based Pricing

## Objective

This task focuses on building predictive models for insurance risk and pricing optimization.

We develop two core components:

1. Claim Severity Model (Regression)
   - Predict TotalClaims for policies with claims > 0

2. Risk-Based Pricing Framework
   - Estimate probability of claim
   - Combine with severity prediction to compute premium

Final goal: build an interpretable and deployable pricing model.

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Dataset

We load the cleaned dataset generated from the DVC pipeline and prepare it for modeling.

In [8]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/insurance_data_clean.csv", sep="|", low_memory=False)

df.head()

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 973382 entries, 0 to 973381
Data columns (total 52 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   UnderwrittenCoverID       973382 non-null  int64  
 1   PolicyID                  973382 non-null  int64  
 2   TransactionMonth          973382 non-null  str    
 3   IsVATRegistered           973382 non-null  bool   
 4   Citizenship               973382 non-null  str    
 5   LegalType                 973382 non-null  str    
 6   Title                     973382 non-null  str    
 7   Language                  973382 non-null  str    
 8   Bank                      831862 non-null  str    
 9   AccountType               934647 non-null  str    
 10  MaritalStatus             965123 non-null  str    
 11  Gender                    963846 non-null  str    
 12  Country                   973382 non-null  str    
 13  Province                  973382 non-null  str    
 14 

# 2. Data Preparation

We prepare the dataset for modeling by:

- Handling missing values
- Creating target variables
- Encoding categorical features
- Splitting train/test datasets

In [9]:
df_model = df.copy()

df_model = df_model[df_model["TotalClaims"] > 0]

df_model["VehicleAge"] = 2026 - df_model["RegistrationYear"]

X = df_model.drop(columns=["TotalClaims"])
y = df_model["TotalClaims"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 3. Feature Encoding

We apply one-hot encoding to categorical variables and pass numeric variables unchanged.

In [10]:
categorical_cols = X.select_dtypes(include=["object"]).columns
numeric_cols = X.select_dtypes(exclude=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

C:\Users\redea\AppData\Local\Temp\ipykernel_13844\2734031632.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns
